###  Hyperparameter Tuning Approaches

In this notebook, I implemented **three different hyperparameter tuning algorithms**:

- **Optuna** — based on **Bayesian optimization**
- **Randomized Search**
- **Grid Search**

Each of these tuning methods can be effective depending on the problem and dataset characteristics.

Since our current dataset is **synthetic**, it is important to try all three approaches to observe which one performs best during training. This provides us with insight into their behavior under controlled conditions.

Later, when we work with **real-world data**, we should also apply all three tuning methods to identify the best hyperparameters and achieve a well-optimized model.


### Import libs

In [3]:
import pandas as pd
import json
import optuna
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from scipy.stats import randint, uniform
from sklearn.model_selection import GridSearchCV, StratifiedKFold,RandomizedSearchCV
from sklearn.metrics import roc_auc_score
import numpy as np

In [4]:
train = pd.read_pickle('../data/train_set.pkl')

In [5]:
with open('../data/selected_features.json', 'r') as file:
    features = json.load(file)

In [6]:
# Some data cleaning
train = train.astype(float)
train.columns = train.columns.str.replace(r'[\[\]<>]', '_', regex=True)

In [7]:
X_train = train[features]
y_train = train['Target']

In [8]:
X_train.shape, y_train.shape

((5571, 21), (5571,))

### Bayesian optimization with Optuna

In [9]:
def objective(trial):
    params = {
        "objective": "binary:logistic",
        "eval_metric": "auc",
        "booster": "gbtree",
        "tree_method": "hist", 
        "max_depth": trial.suggest_int("max_depth", 2, 6),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "n_estimators": trial.suggest_int("n_estimators", 100, 1000),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "use_label_encoder": False 
    }

    model = XGBClassifier(**params)

    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="roc_auc")

    return np.mean(scores)

In [10]:
# Start the study
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50, timeout=600)

[I 2026-02-24 22:52:19,765] A new study created in memory with name: no-name-e7428de1-a4b5-4d0b-9791-5ea1b12302b5
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:52:19] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:52:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:52:20] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } 

In [11]:
# Output results
print("Best ROC AUC score:", study.best_value)
print("Best hyperparameters:", study.best_params)

Best ROC AUC score: 0.7418208788604885
Best hyperparameters: {'max_depth': 3, 'learning_rate': 0.01897663493447101, 'n_estimators': 809, 'subsample': 0.9212084209431806, 'colsample_bytree': 0.779104547835769, 'gamma': 3.9400570531930557, 'reg_alpha': 2.1737849823102315e-06, 'reg_lambda': 0.013549119319063418}


## Randomised Rearch

In [12]:
param_dist = {
    'max_depth': randint(2, 6),
    'learning_rate': uniform(0.01, 0.3),
    'n_estimators': randint(100, 1000),
    'subsample': uniform(0.6, 0.4),
    'colsample_bytree': uniform(0.6, 0.4),
    'gamma': uniform(0, 5),
    'reg_alpha': uniform(0, 10),
    'reg_lambda': uniform(0, 10)
}

model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist'
)

cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)


In [13]:
random_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_dist,
    n_iter=50,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

In [14]:
random_search.fit(X_train, y_train)

Fitting 3 folds for each of 50 candidates, totalling 150 fits


/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:11] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'colsample_bytree': <scipy.stats....t 0x121dee0d0>, 'gamma': <scipy.stats....t 0x120bd7100>, 'learning_rate': <scipy.stats....t 0x1209d3620>, 'max_depth': <scipy.stats....t 0x1209d2e40>, ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",50
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` fo

In [15]:
print("Best ROC AUC:", random_search.best_score_)
print("Best parameters:", random_search.best_params_)

Best ROC AUC: 0.7370081265990703
Best parameters: {'colsample_bytree': np.float64(0.6027808522124762), 'gamma': np.float64(2.553736512887829), 'learning_rate': np.float64(0.1352233009446337), 'max_depth': 2, 'n_estimators': 324, 'reg_alpha': np.float64(1.198653673336828), 'reg_lambda': np.float64(3.3761517140362796), 'subsample': np.float64(0.9771638815650077)}


## Grid Search

In [16]:
param_grid = {
    'max_depth': [2, 4, 6, 3],
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [100, 300, 500],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'gamma': [0, 1, 5]
}

model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='auc',
    use_label_encoder=False,
    tree_method='hist'
)

In [17]:
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=cv,
    verbose=1,
    n_jobs=-1
)

In [18]:
grid_search.fit(X_train, y_train)

Fitting 3 folds for each of 972 candidates, totalling 2916 fits


/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc/Desktop/Diplom/.venv/lib/python3.14/site-packages/xgboost/training.py:200: UserWarning: [22:53:15] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/yurih/Library/CloudStorage/OneDrive-Synopsys,Inc

,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","XGBClassifier...ree=None, ...)"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'colsample_bytree': [0.7, 0.9, ...], 'gamma': [0, 1, ...], 'learning_rate': [0.01, 0.1, ...], 'max_depth': [2, 4, ...], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'roc_auc'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide ` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: intControls the verbosity: the higher, the more messages.- >1 : the comput

In [19]:
print("Best ROC AUC:", grid_search.best_score_)
print("Best parameters:", grid_search.best_params_)

Best ROC AUC: 0.7448444222166253
Best parameters: {'colsample_bytree': 1.0, 'gamma': 1, 'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 500, 'subsample': 1.0}


### Save params

In [20]:
with open("../data/best_grid_search_params.json", "w") as f:
    json.dump(grid_search.best_params_, f, indent=4)


In [21]:
with open("../data/best_random_search_params.json", "w") as f:
    json.dump(random_search.best_params_, f, indent=4)

In [22]:
# Save to JSON file
with open("../data/best_optuna_params.json", "w") as f:
    json.dump(study.best_params, f, indent=4)